In [1]:
#!/usr/bin/env python3
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import glob
import re
import csv
import sys
import torch
import tensorflow as tf
from bm3d import bm3d
from bm4d import bm4d
import keras

# ==========================================
# 0. PATHS AND SYSTEM CONFIGURATION
# ==========================================
model_samp = str(240700000)
base_dir = "/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Data_Gen/plot_data/Image_Den_Plot_allM/IM_NEW"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# SCUNet imports
sys.path.insert(0, "/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet")
from models.network_scunet import SCUNet as SCUNet

# FFT imports
sys.path.insert(0, os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/qmc_algo_tools'))
sys.path.insert(0, os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/developer_tools'))
from qmc_algo_tools.density_denoise import DensityFourierFilterErrorCeil

# Patch torch.load for CPU map_location fallback
_original_torch_load = torch.load
def torch_load_cpu(*args, **kwargs):
    if 'map_location' not in kwargs:
        kwargs['map_location'] = torch.device('cpu')
    return _original_torch_load(*args, **kwargs)
torch.load = torch_load_cpu

keras.config.enable_unsafe_deserialization()
torch.serialization.add_safe_globals([SCUNet])

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def encode_voxel_to_rgb_global(vol_3d):
    """Normalizes the ENTIRE 3D volume, preventing slice-by-slice background amplification."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    
    normed = (vol_3d - v_min) / (v_max - v_min)
    rgb_volume = np.stack([normed]*3, axis=-1).astype(np.float32)
    return rgb_volume, v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    """Restores the 3D volume using the global scalars."""
    gray = rgb_volume[:, :, :, 0]
    return gray * (v_max - v_min) + v_min

def transform(density, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return (density - density_ref) / np.sqrt(np.abs(density_ref) + 1e-8)
    return density

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return density_ref + np.sqrt(np.abs(density_ref) + 1e-8) * density_trans
    return density_trans

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def denoise_with_scunet(rgb_image_np, model, device):
    """Helper to process a single 2D slice through SCUNet."""
    img = np.clip(rgb_image_np.astype(np.float32), 0, 1)
    img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float().unsqueeze(0).to(device)
    with torch.no_grad():
        output_tensor = model(img_tensor)
    output_np = output_tensor.squeeze().cpu().detach().numpy()
    if output_np.ndim == 3:
        output_np = np.transpose(output_np, (1, 2, 0))
    return np.clip(output_np, 0, 1)

def run_model_inference(test_d, dft_d, model_name, models_dict):
    """Routes the input data to the appropriate model logic."""
    # Models using 2D RGB Slicing
    if model_name in ['scunet_pre', 'scunet_trained', 'scunet_ft', 'CAE', 'bm3d']:
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(test_d)
        denoised_rgb = np.zeros_like(rgb_vol)

        if model_name == 'bm3d':
            sigma = np.sqrt(1.0 / 100)
            for i in range(test_d.shape[0]):
                denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
                denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)

        elif model_name.startswith('scunet'):
            model = models_dict[model_name]
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            model.to(device)
            for i in range(test_d.shape[0]):
                denoised_rgb[i] = denoise_with_scunet(rgb_vol[i], model, device)

        elif model_name == 'CAE':
            denoised_rgb = models_dict['CAE'].predict(rgb_vol, verbose=0)
            
        return decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    # 3D Direct Models
    elif model_name == 'CAE_3D':
        input_reshaped = test_d[np.newaxis, ..., np.newaxis]
        dft_reshaped = dft_d[np.newaxis, ..., np.newaxis]
        input_residual = transform(input_reshaped, dft_reshaped, 'residual_noise')
        pred_residual = models_dict['CAE_3D'].predict(input_residual, verbose=0)
        denoised_raw = inverse_transform(pred_residual, dft_reshaped, 'residual_noise')
        return denoised_raw[0, ..., 0]

    elif model_name == 'bm4d':
        return bm4d(test_d, sigma_psd=1/10000)

    elif model_name == 'fft':
        dm = DensityFourierFilterErrorCeil(density_ref=dft_d, filter_mode='augment')
        return dm.denoise(test_d)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

# ==========================================
# 3. EVALUATION
# ==========================================
def evaluate(noisy_d, denoised_d, ref_d, model_name):
    """Enforces physics constraints & standardizes arrays before calculating metrics."""
    # Enforce non-negative density
    denoised_d = np.maximum(denoised_d, 0.0)
    
    # Force EXACT 8-electron sum
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    
    jsd_score = D_JS(denoised_d, ref_d)
    l2_norm = np.linalg.norm(denoised_d - ref_d)

    print(f"   ↳ 2-norm: {l2_norm:.4f} | JSD: {jsd_score:.6e}")
    return denoised_d, jsd_score

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data...")
    with h5py.File(ref_path, 'r') as f: 
        ref_d = f['density'][:]
        ref_d = ref_d * (8.0 / np.sum(ref_d)) # Pre-normalize Reference
        
    with h5py.File(dft_path, 'r') as f: 
        dft_d = f['density'][:]

    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    print(f"Found {len(noisy_files)} noisy files.")

    # Select which models to run
    models_to_run = [
        'scunet_pre', 'scunet_trained', 'scunet_ft', 
        'CAE', 'CAE_3D', 'bm3d', 'bm4d', 'fft'
    ]
    
    print("Pre-loading Models into Memory...")
    models_dict = {}
    
    if 'scunet_pre' in models_to_run:
        m = SCUNet(in_nc=3, config=[4, 4, 4, 4, 4, 4, 4], dim=64)
        m.load_state_dict(torch.load('/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet/model_zoo/scunet_color_25.pth', map_location='cpu'))
        m.eval()
        models_dict['scunet_pre'] = m
        
    if 'scunet_trained' in models_to_run:
        m = torch.load(f'/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/Scunet_trained_Models/{model_samp}_scunet_trained', map_location='cpu', weights_only=False)
        m.eval()
        models_dict['scunet_trained'] = m
        
    if 'scunet_ft' in models_to_run:
        m = torch.load(f'/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/Scunet_FT_Models/{model_samp}_scunet_FT', map_location='cpu', weights_only=False)
        m.eval()
        models_dict['scunet_ft'] = m
        
    if 'CAE' in models_to_run:
        models_dict['CAE'] = keras.models.load_model(f'/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/CAE_img_Models/{model_samp}_2d_CAE_IMG_enc.keras')
        
    if 'CAE_3D' in models_to_run:
        models_dict['CAE_3D'] = tf.keras.models.load_model('/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Density_Models/UNET_3D_Models/residual_denoiser_40M_Blob_NODFT.keras')

    # Run Models Loop
    for model_name in models_to_run:
        print(f"\n=== Running {model_name} ===")
        for noisy_path in noisy_files:
            match = re.search(r"(\d+)\.h5$", noisy_path)
            if not match: continue
            sample_num = int(match.group(1))

            with h5py.File(noisy_path, 'r') as file:
                test_d = file['density'][:]

            print(f" -> Processing Sample {sample_num}...")
            
            # Dispatch
            raw_denoised = run_model_inference(test_d, dft_d, model_name, models_dict)
            
            # Standardize & Evaluate
            clean_denoised, jsd = evaluate(test_d, raw_denoised, ref_d, model_name)

            # Save Output
            out_file = os.path.join(base_dir, f"density_sample_{sample_num}_{model_name}.npy")
            np.save(out_file, clean_denoised)

    generate_final_plots(base_dir, ref_d)


# ==========================================
# 5. UNIFIED PLOTTING & VISUALIZATION
# ==========================================
def generate_final_plots(base_dir, ref_d):
    print("\n=== Generating Master Plots ===")
    eps = 1e-6
    
    # Load VMC Base
    vmc_file = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean_*.h5")))[0]
    with h5py.File(vmc_file, 'r') as f:
        vmc_d = f['density'][:] * (8 / np.sum(f['density'][:]))

    # Discover and Load Models
    npy_files = glob.glob(os.path.join(base_dir, "density_sample_*_*.npy"))
    model_dict = {}
    for fpath in npy_files:
        name = os.path.basename(fpath).lower()
        if "nature" in name: continue
        
        data = np.load(fpath)
        data = data * (8 / np.sum(data))
        
        # Determine Tag
        if "bm3d" in name: model_dict["bm3d"] = data
        elif "bm4d" in name: model_dict["bm4d"] = data
        elif "cae_3d" in name: model_dict["unet3d"] = data
        elif "cae" in name: model_dict["CAE"] = data
        elif "scunet_pre" in name: model_dict["unet_pre"] = data
        elif "scunet_ft" in name: model_dict["unet_ft"] = data
        elif "scunet_trained" in name: model_dict["unet_trained"] = data
        elif "fft" in name: model_dict["fft"] = data

    all_data = [
        ("VMC", vmc_d), 
        ("SCUNet Pre", model_dict.get("unet_pre")),
        ("BM3D", model_dict.get("bm3d")), 
        ("BM4D", model_dict.get("bm4d")), 
        ("SCUNet FT", model_dict.get("unet_ft")),
        ("SCUNet Trained", model_dict.get("unet_trained")), 
        ("CAE", model_dict.get("CAE")), 
        ("UNet (3D)", model_dict.get("unet3d")), 
        ("FFT (3D)", model_dict.get("fft"))
    ]

    # ------ BAR CHART ------
    ref_jsd_names, ref_jsd_scores = [], []
    for name, data in all_data:
        if data is not None:
            ref_jsd_names.append(name)
            ref_jsd_scores.append(D_JS(data, ref_d))

    fig, ax_bar = plt.subplots(figsize=(10, 6))
    bars = ax_bar.bar(ref_jsd_names, ref_jsd_scores, color='cornflowerblue', edgecolor='black', zorder=3)
    ax_bar.set_ylabel('JSD', fontsize=14)
    ax_bar.set_xticks(range(len(ref_jsd_names)))
    ax_bar.set_xticklabels(ref_jsd_names, rotation=45, ha='right', fontsize=12)
    ax_bar.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

    max_y = max(ref_jsd_scores)
    for bar in bars:
        yval = bar.get_height()
        ax_bar.text(bar.get_x() + bar.get_width() / 2, yval + (max_y * 0.02), 
                    f'{yval:.2e}', ha='center', va='bottom', fontsize=10)
    
    ax_bar.set_ylim(0, max_y * 1.2)
    plt.savefig("Image_jsd_comparison.pdf", format='pdf', bbox_inches='tight', dpi=300)
    plt.close()
    
    # ------ 5x2 GRID PLOT ------
    all_data_grid = [
        ("VMC", vmc_d), ("Reference", ref_d),
        ("SCUNet Pre", model_dict.get("unet_pre")), ("UNet (3D)", model_dict.get("unet3d")),
        ("BM3D", model_dict.get("bm3d")), ("SCUNet FT", model_dict.get("unet_ft")),
        ("CAE", model_dict.get("CAE")), ("SCUNet Trained", model_dict.get("unet_trained")),
        ("BM4D", model_dict.get("bm4d")), ("FFT (3D)", model_dict.get("fft"))
    ]

    valid_data = [np.log10(d + eps) for _, d in all_data_grid if d is not None]
    vmin, vmax = min(np.min(d) for d in valid_data), max(np.max(d) for d in valid_data)

    fig, axes = plt.subplots(5, 2, figsize=(12, 20))
    for i, (name, data) in enumerate(all_data_grid):
        ax = axes.flatten()[i]
        if data is None:
            ax.axis('off')
            continue
            
        data_log = np.log10(data + eps)
        central_slice = data_log[:, :, data_log.shape[2] // 2]
        im = ax.imshow(central_slice, origin='lower', vmin=vmin, vmax=vmax, cmap="inferno")
        
        title_str = name if name == "Reference" else f"{name}\nJSD: {D_JS(data, ref_d):.2e}"
        ax.set_title(title_str, fontsize=16, fontweight='bold')  
        ax.axis('off')

    fig.subplots_adjust(top=0.92, bottom=0.05, hspace=0.35, wspace=0.1, right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.03, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='log10(Density)')
    plt.savefig("density_model_comparison.pdf", format="pdf", bbox_inches="tight", dpi=300)
    plt.close()
    
    print("\nExecution Complete. Data arrays and PDF plots have been generated.")

if __name__ == "__main__":
    main()

2026-05-27 10:55:27.354704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779904527.371180 1010971 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779904527.376596 1010971 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779904527.391226 1010971 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779904527.391241 1010971 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779904527.391243 1010971 computation_placer.cc:177] computation placer alr

Loading Base Data...
Found 1 noisy files.
Pre-loading Models into Memory...
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW

I0000 00:00:1779904703.834709 1010971 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 872 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0



=== Running scunet_pre ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0110 | JSD: 3.956259e-02

=== Running scunet_trained ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0023 | JSD: 4.810113e-03

=== Running scunet_ft ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0022 | JSD: 4.186588e-03

=== Running CAE ===
 -> Processing Sample 40960...


I0000 00:00:1779904720.767940 1012463 service.cc:152] XLA service 0x7f01d8004740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779904720.767966 1012463 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2026-05-27 10:58:40.786709: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779904721.022748 1012463 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1779904721.705768 1012463 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   ↳ 2-norm: 0.0042 | JSD: 1.202946e-02

=== Running CAE_3D ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0012 | JSD: 1.090450e-03

=== Running bm3d ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0036 | JSD: 5.028839e-03

=== Running bm4d ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0016 | JSD: 1.793877e-03

=== Running fft ===
 -> Processing Sample 40960...
   ↳ 2-norm: 0.0006 | JSD: 3.765948e-04

=== Generating Master Plots ===


[DEBUG] Loaded backend module://matplotlib_inline.backend_inline version unknown.
[DEBUG] Loaded backend module://matplotlib_inline.backend_inline version unknown.
[DEBUG] findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0.
[DEBUG] findfont: score(FontEntry(fname='/global/u2/k/kberard/environments/SCGSR/lib/python3.12/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizTwoSymBol.ttf', name='STIXSizeTwoSym', style='normal', variant='normal', weight=700, stretch='normal', size='scalable')) = 10.335
[DEBUG] findfont: score(FontEntry(fname='/global/u2/k/kberard/environments/SCGSR/lib/python3.12/site-packages/matplotlib/mpl-data/fonts/ttf/STIXSizFiveSymReg.ttf', name='STIXSizeFiveSym', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
[DEBUG] findfont: score(FontEntry(fname='/global/u2/k/kberard/environments/SCGSR/lib/python3.12/site-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSerif-Bold.ttf', name='DejaV


Execution Complete. Data arrays and PDF plots have been generated.
